In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import getpass
import requests

import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

In [12]:
# Navigate to project workspace
os.chdir('/workspaces/DSE3101-Project')

# Verify correct directory location 
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder to obtain zip file that contains raw data
os.chdir('backend')
current_dir = os.getcwd()
print(f"Latest directory: {current_dir}")
# In your notebook or another script
from hdb_predictor import predict_price_user, predict_price_listing

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'requirements_fixed.txt', 'Final Web Scraper.ipynb', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']
Latest directory: /workspaces/DSE3101-Project/backend


In [13]:
# Navigate to project workspace
os.chdir('/workspaces/DSE3101-Project')

# Verify correct directory location 
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder to obtain zip file that contains raw data
os.chdir('data/raw')
current_dir = os.getcwd()
print(f"Latest directory: {current_dir}")

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'requirements_fixed.txt', 'Final Web Scraper.ipynb', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']
Latest directory: /workspaces/DSE3101-Project/data/raw


In [14]:
listings_df = pd.read_csv('propertyguru_listings_final.csv')

In [15]:
listings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12192 entries, 0 to 12191
Data columns (total 29 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   onemap_full_address                12192 non-null  str    
 2   postal_code                        12185 non-null  float64
 3   price_detail                       12192 non-null  str    
 4   hdb_town                           12192 non-null  str    
 5   room_count                         12192 non-null  float64
 6   floor_area_sqm                     12192 non-null  float64
 7   flat_type                          12192 non-null  str    
 8   lease_commence_date                12192 non-null  float64
 9   remaining_lease                    12192 non-null  float64
 10  flat_age_at_sale                   12192 non-null  float64
 11  sold_year                          12192 non-null  int64  
 12  s

In [16]:
listings_df['predicted_price'] = listings_df.apply(predict_price_listing, axis=1)

In [17]:
listings_df.head()

,listing_url,onemap_full_address,postal_code,price_detail,hdb_town,room_count,floor_area_sqm,flat_type,lease_commence_date,remaining_lease,...,nearest_hawker_distance_m,num_mrt_within_1000m,num_clinic_within_1000m,num_park_within_1000m,num_community_club_within_1000m,num_hawker_within_1000m,num_amenities_within_1000m,nearest_mrt_name,nearest_community_club_name,predicted_price
0,https://www.propertyguru.com.sg/listing/hdb-fo...,903 TAMPINES AVENUE 4 TAMPINES PALMSVILLE SING...,520903.0,"S$ 880,000",Tampines,4.0,133.037096,4 ROOM,1983.0,56.0,...,823.0,2.0,32.0,1.0,2.0,1.0,36.0,TAMPINES MRT STATION,Tampines Central CC,1.007045e+06
1,https://www.propertyguru.com.sg/listing/hdb-fo...,32 BEDOK SOUTH AVENUE 2 SINGAPORE 460032,460032.0,"S$ 799,999",Bedok,4.0,117.986810,4 ROOM,1977.0,50.0,...,350.9,1.0,22.0,7.0,5.0,4.0,38.0,BEDOK MRT STATION,Bedok CC,8.590412e+05
2,https://www.propertyguru.com.sg/listing/hdb-fo...,224 JURONG EAST STREET 21 SINGAPORE 600224,600224.0,"S$ 528,000",Jurong East,4.0,90.952037,4 ROOM,1984.0,57.0,...,225.8,1.0,22.0,4.0,1.0,2.0,29.0,CHINESE GARDEN MRT STATION,Yuhua CC,6.056204e+05
3,https://www.propertyguru.com.sg/listing/hdb-fo...,302 CLEMENTI AVENUE 4 CLEMENTI MEADOWS SINGAPO...,120302.0,"S$ 578,000",Clementi,4.0,91.973970,4 ROOM,1978.0,51.0,...,932.0,1.0,11.0,6.0,1.0,1.0,19.0,CLEMENTI MRT STATION,Clementi CC,6.791702e+05
4,https://www.propertyguru.com.sg/listing/hdb-fo...,94 DAWSON ROAD SKYPARC @ DAWSON SINGAPORE 141094,141094.0,"S$ 1,400,000",Queenstown,4.0,87.979141,4 ROOM,2020.0,93.0,...,983.2,2.0,10.0,0.0,1.0,1.0,12.0,QUEENSTOWN MRT STATION,Leng Kee CC,1.106478e+06


In [18]:
results = []
for _, row in listings_df.iterrows():
    try:
        actual = float(str(row['price_detail']).replace('S$', '').replace(',', '').strip())  # ← fix
        error = row['predicted_price'] - actual
        pct_error = (error / actual) * 100
        results.append({
            'town':            row['hdb_town'],
            'flat_type':       row['flat_type'],
            'floor_area':      row['floor_area_sqm'],
            'remaining_lease': row['remaining_lease'],
            'actual_price':    actual,
            'predicted_price': row['predicted_price'],
            'error':           error,
            'pct_error':       pct_error,
            'abs_pct_error':   abs(pct_error),
        })
    except Exception as e:
        print(f"Skipped row: {e}")

results_df = pd.DataFrame(results)
print(f"Rows collected: {len(results_df)}")

Rows collected: 12192


In [19]:

print(f"\n=== Prediction vs Listing Price ===")
print(f"Listings tested:       {len(results_df)}")
print(f"Mean error:            ${results_df['error'].mean():+,.0f}  (+ = overpredicting, - = underpredicting)")
print(f"Median abs error:      ${results_df['error'].abs().median():,.0f}")
print(f"MAE:                   ${results_df['error'].abs().mean():,.0f}")
print(f"MAPE:                  {results_df['abs_pct_error'].mean():.1f}%")

print(f"\n=== Error by Town (mean % error) ===")
town_errors = results_df.groupby('town')['pct_error'].mean().sort_values()
print(town_errors.round(1).to_string())

print(f"\n=== Error by Flat Type (mean % error) ===")
type_errors = results_df.groupby('flat_type')['pct_error'].mean().sort_values()
print(type_errors.round(1).to_string())



=== Prediction vs Listing Price ===
Listings tested:       12192
Mean error:            $+81,500  (+ = overpredicting, - = underpredicting)
Median abs error:      $90,237
MAE:                   $119,611
MAPE:                  40.8%

=== Error by Town (mean % error) ===
town
Central Area         0.2
Punggol              2.2
Marine Parade        2.9
Hougang              4.0
Bukit Timah          6.4
Bishan               6.8
Toa Payoh            8.3
Clementi             8.3
Bukit Panjang        8.6
Bukit Merah          9.1
Queenstown           9.8
Serangoon           10.0
Kallang/Whampoa     10.6
Jurong East         10.6
Bukit Batok         14.8
Pasir Ris           15.9
Sembawang           16.4
Ang Mo Kio          17.4
Geylang             18.5
Woodlands           18.6
Jurong West         19.6
Bedok               20.5
Sengkang            24.5
Choa Chu Kang       45.2
Tampines           144.3
Yishun             229.7

=== Error by Flat Type (mean % error) ===
flat_type
2 ROOM    13.9
3 ROOM

In [20]:
lol

NameError: name 'lol' is not defined

In [ ]:
import joblib
import numpy as np
import pandas as pd
from datetime import datetime

# Load both models
artifacts_v1 = joblib.load('hdb_model_artifacts.pkl')

# ── V1 artifacts ──────────────────────────────────────────
xgb_model_v1        = artifacts_v1['xgb_model']
encoders_v1         = artifacts_v1['encoders']
TREE_FEATURES_v1    = artifacts_v1['TREE_FEATURES']
categorical_cols_v1 = artifacts_v1['categorical_cols']
hdb_df              = artifacts_v1['hdb_df'] 
X_train_v1          = artifacts_v1['X_train']      # needed for fallback medians

In [ ]:
# ── Shared constants ──────────────────────────────────────
TOWN_PREMIUM = {
    'MARINE PARADE': 1.17, 'QUEENSTOWN': 1.15, 'CLEMENTI': 1.11,
    'BUKIT MERAH':   1.10, 'BUKIT PANJANG': 1.10, 'HOUGANG': 1.09,
    'BEDOK':         1.08, 'KALLANG/WHAMPOA': 1.08, 'JURONG EAST': 1.07,
    'BUKIT BATOK':   1.07, 'CENTRAL AREA': 1.07, 'TAMPINES': 1.07,
    'JURONG WEST':   1.06, 'CHOA CHU KANG': 1.06, 'SERANGOON': 1.06,
    'TOA PAYOH':     1.06, 'PASIR RIS': 1.05, 'ANG MO KIO': 1.05,
    'WOODLANDS':     1.05, 'SENGKANG': 1.05, 'YISHUN': 1.05,
    'BISHAN':        1.04, 'PUNGGOL': 1.04, 'BUKIT TIMAH': 1.03,
    'GEYLANG':       1.03, 'SEMBAWANG': 1.02,
}
DEFAULT_PREMIUM = 1.07

FLAT_TYPE_ADJUSTMENT = {
    '2 ROOM': 1.035, '3 ROOM': 1.014, 'EXECUTIVE': 1.008,
    '4 ROOM': 0.998, '5 ROOM': 0.993, 'MULTI-GENERATION': 1.000,
}

# ── V1 prediction ─────────────────────────────────────────
def predict_v1(town, flat_type, floor_area, sold_year=None,
               remaining_lease=None):
    if sold_year is None:
        sold_year = datetime.now().year

    # FIXED: Use proper boolean indexing to avoid index alignment error
    mask1 = (hdb_df['town'] == town)
    mask2 = (hdb_df['flat_type'] == flat_type)
    mask3 = (hdb_df['sold_year'] >= 2022)
    
    ref = hdb_df[mask1 & mask2 & mask3]
    
    if ref.empty:
        ref = hdb_df[mask1 & mask2]
    if ref.empty:
        ref = hdb_df[mask1]
    if ref.empty:
        ref = hdb_df.copy()

    if remaining_lease is None:
        remaining_lease = int(ref['remaining_lease'].median())

    flat_age            = 99 - remaining_lease
    lease_commence_date = sold_year - flat_age

    input_dict = {
        'town':                              town,
        'flat_type':                         flat_type,
        'storey_range':                      ref['storey_range'].mode()[0],
        'floor_area_sqm':                    floor_area,
        'flat_model':                        ref['flat_model'].mode()[0],
        'lease_commence_date':               lease_commence_date,
        'remaining_lease':                   remaining_lease,
        'sold_year':                         sold_year,
        'storey_mid':                        ref['storey_mid'].median(),
        'storey_category':                   ref['storey_category'].mode()[0],
        'region':                            ref['region'].mode()[0],
        'is_mature_estate':                  int(ref['is_mature_estate'].mode()[0]),
        'nearest_mrt_distance_m':            ref['nearest_mrt_distance_m'].median(),
        'nearest_clinic_distance_m':         ref['nearest_clinic_distance_m'].median(),
        'nearest_park_distance_m':           ref['nearest_park_distance_m'].median(),
        'nearest_community_club_distance_m': ref['nearest_community_club_distance_m'].median(),
        'nearest_hawker_distance_m':         ref['nearest_hawker_distance_m'].median(),
        'num_mrt_within_1000m':              ref['num_mrt_within_1000m'].median(),
        'num_clinic_within_1000m':           ref['num_clinic_within_1000m'].median(),
        'num_park_within_1000m':             ref['num_park_within_1000m'].median(),
        'num_community_club_within_1000m':   ref['num_community_club_within_1000m'].median(),
        'num_hawker_within_1000m':           ref['num_hawker_within_1000m'].median(),
        'num_amenities_within_1000m':        ref['num_amenities_within_1000m'].median(),
        'flat_age_at_sale':                  flat_age,
    }

    # Fill any remaining features with training medians
    for col in TREE_FEATURES_v1:
        if col not in input_dict:
            input_dict[col] = X_train_v1[col].median()

    input_df = pd.DataFrame([input_dict])[TREE_FEATURES_v1]
    for col in categorical_cols_v1:
        if col in input_df.columns:
            try:
                input_df[col] = encoders_v1[col].transform(input_df[col].astype(str))
            except ValueError:
                input_df[col] = X_train_v1[col].mode()[0]

    log_pred   = xgb_model_v1.predict(input_df)[0]
    transacted = np.exp(log_pred) * floor_area * 203.6
    return round(transacted * TOWN_PREMIUM.get(town, DEFAULT_PREMIUM) *
                 FLAT_TYPE_ADJUSTMENT.get(flat_type, 1.0), 2)

# ── Compare both models ───────────────────────────────────
town, flat_type, floor_area = 'QUEENSTOWN', '4 ROOM', 98
v1 = predict_v1(town, flat_type, floor_area, remaining_lease=68)

print(f"V1 (random-split, full features): ${v1:,.0f}")

V1 (random-split, full features): $897,922


In [ ]:
# Run predictions
results = []
for _, row in listings_df.iterrows():
    try:
        predicted = predict_v1(
            town=row['hdb_town'],
            flat_type=row['flat_type'],
            floor_area=float(row['floor_area_sqm']),
            sold_year=2026,
            remaining_lease=float(row['remaining_lease'])
        )
        actual = float(row['price_details'])
        error = predicted - actual
        pct_error = (error / actual) * 100
        results.append({
            'town': row['hdb_town'],
            'flat_type': row['flat_type'],
            'floor_area': row['floor_area_sqm'],
            'remaining_lease': row['remaining_lease'],
            'actual_price': actual,
            'predicted_price': predicted,
            'error': error,
            'pct_error': pct_error,
            'abs_pct_error': abs(pct_error),
        })
    except Exception as e:
        pass

results_df = pd.DataFrame(results)

print(f"\n=== Prediction vs Listing Price ===")
print(f"Listings tested:       {len(results_df)}")
print(f"Mean error:            ${results_df['error'].mean():+,.0f}  (+ = overpredicting, - = underpredicting)")
print(f"Median abs error:      ${results_df['error'].abs().median():,.0f}")
print(f"MAE:                   ${results_df['error'].abs().mean():,.0f}")
print(f"MAPE:                  {results_df['abs_pct_error'].mean():.1f}%")

print(f"\n=== Error by Town (mean % error) ===")
town_errors = results_df.groupby('town')['pct_error'].mean().sort_values()
print(town_errors.round(1).to_string())

print(f"\n=== Error by Flat Type (mean % error) ===")
type_errors = results_df.groupby('flat_type')['pct_error'].mean().sort_values()
print(type_errors.round(1).to_string())

KeyboardInterrupt: 

In [ ]:
# ── Compare both models ───────────────────────────────────
town, flat_type, floor_area = 'QUEENSTOWN', '3 ROOM', 85
v1 = predict_v1(town, flat_type, floor_area, remaining_lease=68)

print(f"V1 (random-split, full features): ${v1:,.0f}")

V1 (random-split, full features): $767,062


In [ ]:
# Apply V1 predictions to a dataframe
listings_df['predicted_price_v1'] = listings_df.apply(
    lambda row: predict_v1(
        town             = row['hdb_town'],
        flat_type        = row['flat_type'],
        floor_area       = row['floor_area_sqm'],
        remaining_lease  = row['remaining_lease']
    ),
    axis=1
)

print(listings_df[['hdb_town', 'flat_type', 'floor_area_sqm', 
          'remaining_lease', 'predicted_price_v1']].head())

KeyboardInterrupt: 